### Please run this file with Intake.txt and Patient Nutrition Needs.txt
- Intake.txt is json file loaded from patient diet recall pdf file
- Patient Nutrition Needs.txt is text from sheet DRI Progress Tracker of:
- https://docs.google.com/spreadsheets/d/1_ufMhRzRLVcmoP1_F_Q6SxrkNSDamZYM7sn2yOrB2c4/edit?gid=1382032586#gid=1382032586

In [1]:
# find food list from intake json file

import json

# Load JSON data from file
file_path = "Intake.txt"  # Update with the correct path
with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Extract food descriptions
food_items = []
for meal in data["patient"]["diet_recall"]:
    for food in meal["food"]:
        food_items.append(food["description"])

# Print the extracted food items
print("Food Items:")
print("\n".join(food_items))


Food Items:
Raisin Bran
Apple juice
Fresh peach
Ground beef
Mushroom stew
Rice
Green beans
Water
Pretzels
Chocolate
Spaghetti
Ground beef
Water
Single Malt


In [2]:
# find nutrition fact for each food, but exclude water

import pandas as pd
import requests

# Dictionary to collect nutrient data
nutrient_data = {}

# API setup
API_KEY = "A2cUE0WUknfVIuJGdkebUCcKjddw1RD0bpAny1SC"
search_url = "https://api.nal.usda.gov/fdc/v1/foods/search"
headers = {"Content-Type": "application/json"}

# Iterate over food items
for food in food_items:
    if food.lower() == 'water': 
        continue  
        
    payload = {
        "query": food,
        "requireAllWords": True,
    }

    try:
        response = requests.post(f"{search_url}?api_key={API_KEY}", json=payload, headers=headers)
        response.raise_for_status()
        data = response.json()
        foods = data.get("foods", [])
        if not foods:
            continue

        first_fdc_id = foods[0]["fdcId"]
        detail_url = f"https://api.nal.usda.gov/fdc/v1/food/{first_fdc_id}?api_key={API_KEY}"
        response = requests.get(detail_url)
        response.raise_for_status()

        food_data = response.json()
        food_name = food_data.get("description", food)

        for nutrient in food_data.get("foodNutrients", []):
            name = nutrient["nutrient"]["name"]
            amount = nutrient.get("amount")
            unit = nutrient["nutrient"]["unitName"]
            label = f"{amount} {unit}" if amount is not None else "N/A"

            if name not in nutrient_data:
                nutrient_data[name] = {}
            nutrient_data[name][food] = label

    except Exception as e:
        print(f"Error processing {food}: {e}")
        continue
# print(nutrient_data)
# Create DataFrame
df = pd.DataFrame(nutrient_data).T
df.index.name = "Nutrition"
df.to_csv("nutrition_table.csv")
df

,Raisin Bran,Apple juice,Fresh peach,Ground beef,Mushroom stew,Rice,Green beans,Pretzels,Chocolate,Spaghetti,Single Malt
Nutrition,,,,,,,,,,,
Vitamin A,153.0 µg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"Fatty acids, total saturated",0.0 g,0.0 g,0.0 g,10.71 g,1.39 g,0.0 g,0.0 g,0.0 g,22.86 g,0.0 g,2.97 g
"Folate, total",339.0 µg,NaN,NaN,NaN,NaN,133.0 µg,NaN,NaN,NaN,214.0 µg,NaN
"Fatty acids, total monounsaturated",0.0 g,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0 g,NaN
"Vitamin C, total ascorbic acid",0.0 mg,NaN,0.0 mg,NaN,NaN,0.0 mg,3.0 mg,NaN,0.0 mg,0.0 mg,0.0 mg
Niacin,8.475 mg,NaN,NaN,NaN,NaN,3.556 mg,NaN,NaN,NaN,7.143 mg,NaN
"Calcium, Ca",34.0 mg,NaN,0.0 mg,18.0 mg,12.0 mg,0.0 mg,17.0 mg,NaN,57.0 mg,0.0 mg,127.0 mg
"Potassium, K",542.0 mg,108.0 mg,NaN,NaN,44.0 mg,89.0 mg,NaN,NaN,NaN,NaN,NaN
Vitamin D (D2 + D3),5.0 µg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# seperate unit and amount for nutrition_table

import pandas as pd
import re

# Load the CSV file
df = pd.read_csv("nutrition_table.csv", index_col=0)

# Select the first food column to extract units from
first_food_col = df.columns[0]

# Initialize UNIT column
unit_column = []

# Process each row to extract unit and strip it from all columns
for index, row in df.iterrows():
    first_val = row[first_food_col]
    if pd.isna(first_val) or not isinstance(first_val, str):
        unit_column.append(None)
        continue

    # Extract numeric value and unit using regex
    match = re.match(r"([-+]?\d*\.\d+|\d+)\s*(\D+)", first_val.strip())
    if match:
        unit = match.group(2).strip()
    else:
        unit = None
    unit_column.append(unit)

# Add the UNIT column
df["UNIT"] = unit_column

# Remove units and convert to numeric values
for col in df.columns[:-1]:  # Skip the UNIT column
    df[col] = df[col].astype(str).str.extract(r"([-+]?\d*\.\d+|\d+)")[0]
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.to_csv("food_nutrition_table.csv")
df

,Raisin Bran,Apple juice,Fresh peach,Ground beef,Mushroom stew,Rice,Green beans,Pretzels,Chocolate,Spaghetti,Single Malt,UNIT
Nutrition,,,,,,,,,,,,
Vitamin A,153.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,µg
"Fatty acids, total saturated",0.000,0.0,0.00,10.71,1.39,0.000,0.00,0.00,22.86,0.000,2.97,g
"Folate, total",339.000,NaN,NaN,NaN,NaN,133.000,NaN,NaN,NaN,214.000,NaN,µg
"Fatty acids, total monounsaturated",0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000,NaN,g
"Vitamin C, total ascorbic acid",0.000,NaN,0.00,NaN,NaN,0.000,3.00,NaN,0.00,0.000,0.00,mg
Niacin,8.475,NaN,NaN,NaN,NaN,3.556,NaN,NaN,NaN,7.143,NaN,mg
"Calcium, Ca",34.000,NaN,0.00,18.00,12.00,0.000,17.00,NaN,57.00,0.000,127.00,mg
"Potassium, K",542.000,108.0,NaN,NaN,44.00,89.000,NaN,NaN,NaN,NaN,NaN,mg
Vitamin D (D2 + D3),5.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,µg


In [4]:
# load food intake json file again to get food intake, convert unit to ml or g

import json
import pandas as pd

# Conversion factors to grams or milliliters (approximate values)
CONVERSIONS = {
    "cup": (240, "ml"),       # ml
    "oz": (29.57, "ml"),      # ml
    "ml": (1, "ml"),          # ml
    "g": (1, "g"),            # g
    "medium": (150, "g")      # g
}

# Fractions to float
FRACTIONS = {
    "¼": 0.25,
    "½": 0.5,
    "¾": 0.75
}

def parse_amount(amount):
    for frac, val in FRACTIONS.items():
        amount = amount.replace(frac, str(val))
    return amount

def convert_to_float(amount):
    try:
        return eval(amount)
    except:
        return None

def convert_unit(amount_str):
    parts = amount_str.strip().split()
    if len(parts) == 2:
        value_str, unit = parts
        parsed_value_str = parse_amount(value_str)
        value = convert_to_float(parsed_value_str)
        if value is not None and unit in CONVERSIONS:
            factor, target_unit = CONVERSIONS[unit]
            return value * factor, target_unit, value
    elif len(parts) == 1 and parts[0] in CONVERSIONS:
        factor, target_unit = CONVERSIONS[parts[0]]
        return factor, target_unit, 1
    return None, None, None

# Load JSON data from file
file_path = "Intake.txt"  # Update with the correct path
with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Process and convert food items
converted_foods = []
for meal in data["patient"]["diet_recall"]:
    for food in meal["food"]:
        amount_str = food["amount"]
        description = food["description"]
        converted_value, target_unit, numeric_value = convert_unit(amount_str)
        if converted_value is not None:
            rounded_value = int(converted_value) if converted_value.is_integer() else round(converted_value) # , 2)
            converted_foods.append({
                "food": description,
                "amount": amount_str,
                "amount_in_ml_or_g": rounded_value
            })
        else:
            converted_foods.append({
                "food": description,
                "amount": amount_str,
                "amount_in_ml_or_g": "unknown"
            })

# Create DataFrame
df = pd.DataFrame(converted_foods)

# Filter out unknowns
df_clean = df[df["amount_in_ml_or_g"] != "unknown"]

# Group and sum by food
df_food_summary = df_clean.groupby("food", as_index=False)["amount_in_ml_or_g"].sum()
df_food_summary.to_csv("food_summary.csv")
df_food_summary

,food,amount_in_ml_or_g
0,Apple juice,120
1,Chocolate,30
2,Fresh peach,150
3,Green beans,60
4,Ground beef,240
5,Mushroom stew,240
6,Pretzels,120
7,Raisin Bran,180
8,Rice,120
9,Single Malt,60


In [5]:
# Sum nutrition 

import pandas as pd

# Load the data
food_summary = pd.read_csv("food_summary.csv")
food_nutrition_table = pd.read_csv("food_nutrition_table.csv")

# get water amount first
water_amount = food_summary.loc[food_summary["food"] == "Water", "amount_in_ml_or_g"].values[0]

# Drop the 'Unnamed: 0' column from food_summary
food_summary = food_summary.drop(columns=["Unnamed: 0"])

# Set 'Nutrition' column as index and drop 'UNIT' temporarily
nutrition_data = food_nutrition_table.set_index("Nutrition").drop(columns=["UNIT"])

# Create a dictionary of scaling factors: {food_name: amount / 100}
scaling_factors = dict(zip(food_summary["food"], food_summary["amount_in_ml_or_g"] / 100))

# Multiply each food column by its corresponding scaling factor (if it exists)
adjusted_nutrition = nutrition_data.copy()
for food in adjusted_nutrition.columns:
    # print('food: ', food)
    if food in scaling_factors:
        adjusted_nutrition[food] *= scaling_factors[food]
    else:
        adjusted_nutrition[food] = 0  # food not consumed that day

# Sum across all food columns to get total nutrient intake
df_nutrition_summary = adjusted_nutrition.sum(axis=1).to_frame(name="Amount")
# Extract the UNIT column from the original food_nutrition_table
unit_column = food_nutrition_table.set_index("Nutrition")["UNIT"]
# Join the unit column with df_nutrition_summary
df_nutrition_summary = df_nutrition_summary.join(unit_column)
# add water back to nutrition list
df_nutrition_summary.loc["Water"] = [water_amount/1000, "liters"]
# save to csv file
df_nutrition_summary.to_csv("nutrition_total_intake.csv")
df_nutrition_summary


,Amount,UNIT
Nutrition,,
Vitamin A,275.4000,µg
"Fatty acids, total saturated",37.6800,g
"Folate, total",1283.4000,µg
"Fatty acids, total monounsaturated",0.0000,g
"Vitamin C, total ascorbic acid",1.8000,mg
Niacin,36.6654,mg
"Calcium, Ca",236.7000,mg
"Potassium, K",1317.6000,mg
Vitamin D (D2 + D3),9.0000,µg


In [6]:
# Re-import required libraries due to code execution state reset
import pandas as pd
import re
import numpy as np

# Re-read the file after reset
file_path = "Patient Nutrition Needs.txt"
with open(file_path, "r") as file:
    lines = file.readlines()

# Helper function to split amount and unit properly
def parse_amount_unit(value):
    if 'low as possible' in value:
        return np.nan, ''
    if ' - ' in value:
        nums = re.findall(r"[\d.,]+", value)
        nums = [float(n.replace(',', '')) for n in nums]
        mean_val = sum(nums) / len(nums)
        unit = value.split()[-1]
        return round(mean_val, 2), unit
    if '(' in value:
        value = value.split('(')[0].strip()
    match = re.match(r"([\d.,]+)\s*([a-zA-Z/]+)", value)
    if match:
        amount, unit = match.groups()
        return float(amount.replace(',', '')), unit
    return np.nan, ''

# Reprocess the file with updated parsing
data = []
category = None
valid_categories = ['Macronutrient', 'Vitamin', 'Mineral']

for line in lines:
    line = line.strip()
    if not line:
        continue
    if any(line == cat or line.startswith(cat + '\t') for cat in valid_categories):
        category = next(cat for cat in valid_categories if line.startswith(cat))
        continue
    if line.startswith('Estimated Daily Caloric Needs'):
        value = line.split('\t')[-1]
        amount, unit = parse_amount_unit(value)
        data.append(['Calory', 'Macronutrient', amount, 'kcal'])
        continue
    parts = line.split('\t')
    if len(parts) == 2:
        nutrition = parts[0].strip()
        value = parts[1].strip()
        amount, unit = parse_amount_unit(value)
        data.append([nutrition, category, amount, unit])

# Create the cleaned DataFrame
df_final = pd.DataFrame(data, columns=['Nutrition', 'Category', 'Need_Amount', 'Need_Unit'])
df_final.to_csv("nutrition_total_needs.csv")
df_final

,Nutrition,Category,Need_Amount,Need_Unit
0,Calory,Macronutrient,3063.00,kcal
1,Carbohydrate,Macronutrient,421.50,grams
2,Total Fiber,Macronutrient,43.00,grams
3,Protein,Macronutrient,60.00,grams
4,Fat,Macronutrient,93.50,grams
5,Saturated fatty acids,Macronutrient,NaN,
6,Trans fatty acids,Macronutrient,NaN,
7,Î±-Linolenic Acid,Macronutrient,1.60,grams
8,Linoleic Acid,Macronutrient,17.00,grams
9,Dietary Cholesterol,Macronutrient,NaN,


In [7]:
# Define the ordered Need_Nutrition list
ordered_need_nutrition = [
    "Calory", "Carbohydrate", "Total Fiber", "Protein", "Fat",
    "Saturated fatty acids", "Trans fatty acids", "Î±-Linolenic Acid", "Linoleic Acid",
    "Dietary Cholesterol", "Total Water", "Vitamin A", "Vitamin C", "Vitamin D",
    "Vitamin B6", "Vitamin E", "Vitamin K", "Thiamin", "Vitamin B12", "Riboflavin",
    "Folate", "Niacin", "Choline", "Pantothenic Acid", "Biotin", "Carotenoids",
    "Calcium", "Chloride", "Chromium", "Copper", "Fluoride", "Iodine", "Iron",
    "Magnesium", "Manganese", "Molybdenum", "Phosphorus", "Potassium", "Selenium",
    "Sodium", "Zinc"
]

# Define the combined mapping as a dictionary
mapping_dict = {
    "Calory": "Energy",
    "Carbohydrate": "Carbohydrate, by difference",
    "Total Fiber": "Fiber, total dietary",
    "Protein": "Protein",
    "Fat": "Total lipid (fat)",
    "Saturated fatty acids": "Fatty acids, total saturated",
    "Trans fatty acids": "Fatty acids, total trans",
    "Î±-Linolenic Acid": "*Not directly available*",
    "Linoleic Acid": "*Not directly available*",
    "Dietary Cholesterol": "Cholesterol",
    "Total Water": "Water",
    "Vitamin A": "Vitamin A",
    "Vitamin C": "Vitamin C, total ascorbic acid",
    "Vitamin D": "Vitamin D (D2 + D3)",
    "Vitamin B6": "Vitamin B-6",
    "Vitamin E": "*Not available*",
    "Vitamin K": "*Not available*",
    "Thiamin": "Thiamin",
    "Vitamin B12": "Vitamin B-12",
    "Riboflavin": "Riboflavin",
    "Folate": "Folate, total",
    "Niacin": "Niacin",
    "Choline": "*Not available*",
    "Pantothenic Acid": "*Not available*",
    "Biotin": "*Not available*",
    "Carotenoids": "*Not available*",
    "Calcium": "Calcium, Ca",
    "Chloride": "*Not available*",
    "Chromium": "*Not available*",
    "Copper": "Copper, Cu",
    "Fluoride": "*Not available*",
    "Iodine": "*Not available*",
    "Iron": "Iron, Fe",
    "Magnesium": "Magnesium, Mg",
    "Manganese": "*Not available*",
    "Molybdenum": "*Not available*",
    "Phosphorus": "Phosphorus, P",
    "Potassium": "Potassium, K",
    "Selenium": "*Not available*",
    "Sodium": "Sodium, Na",
    "Zinc": "Zinc, Zn"
}

# Create ordered DataFrame
ordered_df = pd.DataFrame([
    (nutrient, mapping_dict.get(nutrient, "*Not available*"))
    for nutrient in ordered_need_nutrition
], columns=["Need_Nutrition", "Intake_Nutrition"])

# Save to CSV
ordered_csv_path = "Ordered_Mapped_Nutrients.csv"
ordered_df.to_csv(ordered_csv_path, index=False)
ordered_df

,Need_Nutrition,Intake_Nutrition
0,Calory,Energy
1,Carbohydrate,"Carbohydrate, by difference"
2,Total Fiber,"Fiber, total dietary"
3,Protein,Protein
4,Fat,Total lipid (fat)
5,Saturated fatty acids,"Fatty acids, total saturated"
6,Trans fatty acids,"Fatty acids, total trans"
7,Î±-Linolenic Acid,*Not directly available*
8,Linoleic Acid,*Not directly available*
9,Dietary Cholesterol,Cholesterol


In [8]:
# Load the uploaded CSV files
intake_df = pd.read_csv("nutrition_total_intake.csv")
needs_df = pd.read_csv("nutrition_total_needs.csv")
mapping_df = pd.read_csv("Ordered_Mapped_Nutrients.csv")

# Create mapping dictionary
name_mapping = dict(zip(mapping_df["Intake_Nutrition"], mapping_df["Need_Nutrition"]))

# Map and prepare intake data
intake_df["Nutrition"] = intake_df["Nutrition"].map(name_mapping)
intake_df = intake_df.dropna(subset=["Nutrition"])  # Drop unmapped rows
intake_df = intake_df.rename(columns={"UNIT": "Intake_Unit"})

# Merge with needs data
# merged_df = pd.merge(needs_df, intake_df, on="Nutrition", how="left")
merged_df = pd.merge(needs_df, intake_df, on="Nutrition", how="left").drop(columns=["Unnamed: 0"], errors="ignore")

merged_df["Deviation"] = merged_df["Need_Amount"] - merged_df["Amount"]
# Round all numeric columns to 1 decimal point
numeric_cols = merged_df.select_dtypes(include=["float64", "int64"]).columns
merged_df[numeric_cols] = merged_df[numeric_cols].round(1)
merged_df = merged_df.rename(columns={"Amount": "Intake_Amount"})

# Save the final merged CSV
final_merged_path = "Nutrition_Intake_vs_Needs.csv"
merged_df.to_csv(final_merged_path, index=False)

merged_df


,Nutrition,Category,Need_Amount,Need_Unit,Intake_Amount,Intake_Unit,Deviation
0,Calory,Macronutrient,3063.0,kcal,3805.8,kcal,-742.8
1,Carbohydrate,Macronutrient,421.5,grams,576.2,g,-154.7
2,Total Fiber,Macronutrient,43.0,grams,35.2,g,7.8
3,Protein,Macronutrient,60.0,grams,114.8,g,-54.8
4,Fat,Macronutrient,93.5,grams,123.5,g,-30.0
5,Saturated fatty acids,Macronutrient,NaN,NaN,37.7,g,NaN
6,Trans fatty acids,Macronutrient,NaN,NaN,0.0,g,NaN
7,Î±-Linolenic Acid,Macronutrient,1.6,grams,NaN,NaN,NaN
8,Linoleic Acid,Macronutrient,17.0,grams,NaN,NaN,NaN
9,Dietary Cholesterol,Macronutrient,NaN,NaN,250.8,mg,NaN
